# Nível 2 — Agente de Investigação de PLD

Neste nível será desenvolvido um agente capaz de investigar operações financeiras utilizando ferramentas determinísticas e um modelo de linguagem.

A abordagem separa as responsabilidades:
- Python executa consultas e cálculos sobre os dados;
- as ferramentas fornecem evidências objetivas;
- o LLM interpreta as evidências e produz o parecer final.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)

load_dotenv("../.env")

CAMINHO_DADOS = Path("../dados/dados_nivel_2.json")

print("Arquivo encontrado:", CAMINHO_DADOS.exists())

Arquivo encontrado: True


In [2]:
with open(CAMINHO_DADOS, "r", encoding="utf-8") as arquivo:
    dados_nivel_2 = json.load(arquivo)

taxa_cambio = dados_nivel_2["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados_nivel_2["operacoes"])

print(f"Taxa USD/BRL: {taxa_cambio}")
print(f"Quantidade inicial de registros: {len(df)}")
print(f"Quantidade de clientes: {df['cliente_id'].nunique()}")
print(f"Quantidade de IDs únicos: {df['id'].nunique()}")

display(df.head())

Taxa USD/BRL: 5.4
Quantidade inicial de registros: 322
Quantidade de clientes: 30
Quantidade de IDs únicos: 317


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,


In [3]:
print("Valores ausentes:")
display(df.isna().sum().to_frame("quantidade"))

print("\nIDs duplicados:")
duplicados = (
    df[df.duplicated(subset=["id"], keep=False)]
    .sort_values("id")
)

display(duplicados)

Valores ausentes:


,quantidade
id,0
cliente_id,0
data,7
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0



IDs duplicados:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
63,OP-00040,CLI-005,NaN,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
186,OP-00040,CLI-005,NaN,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
10,OP-00160,CLI-017,2026-03-19,3712.72,BRL,especie,saque,Lumen Industria ME,
143,OP-00160,CLI-017,2026-03-19,3712.72,BRL,especie,saque,Lumen Industria ME,
54,OP-00214,CLI-023,2026-03-20,1335.29,BRL,ted,saque,Mirante Consultoria SA,
253,OP-00214,CLI-023,2026-03-20,1335.29,BRL,ted,saque,Mirante Consultoria SA,
7,OP-00269,CLI-028,2026-05-23,6913.84,BRL,cartao,transferencia_enviada,Farol Distribuidora LTDA,
118,OP-00269,CLI-028,2026-05-23,6913.84,BRL,cartao,transferencia_enviada,Farol Distribuidora LTDA,
248,OP-00272,CLI-028,2026-03-27,6076.89,BRL,boleto,transferencia_enviada,Lumen Servicos SA,
266,OP-00272,CLI-028,2026-03-27,6076.89,BRL,boleto,transferencia_enviada,Lumen Servicos SA,


In [4]:
print("Moedas:")
display(df["moeda"].value_counts())

print("\nCanais:")
display(df["canal"].value_counts())

print("\nTipos:")
display(df["tipo"].value_counts())

Moedas:


moeda
BRL    315
USD      7
Name: count, dtype: int64


Canais:


canal
ted        82
especie    67
pix        61
cartao     58
boleto     54
Name: count, dtype: int64


Tipos:


tipo
deposito                  69
pagamento                 68
transferencia_enviada     68
transferencia_recebida    60
saque                     57
Name: count, dtype: int64

## Limpeza e normalização dos dados

Antes da investigação, os dados são normalizados para evitar que problemas de qualidade interfiram na análise.

São tratados:
- registros duplicados;
- datas ausentes;
- conversão de valores em USD para BRL;
- padronização dos tipos de dados.

Operações sem data conhecida são preservadas, mas não são utilizadas em análises que dependem de agrupamento temporal.

In [5]:
df_limpo = df.copy()

# Remove registros duplicados pelo ID da operação
df_limpo = df_limpo.drop_duplicates(subset=["id"], keep="first").copy()

# Converte a coluna de data
df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")

# Converte todos os valores para BRL
df_limpo["valor_brl"] = df_limpo.apply(
    lambda linha: (
        linha["valor"] * taxa_cambio
        if linha["moeda"] == "USD"
        else linha["valor"]
    ),
    axis=1
)

print(f"Registros antes da limpeza: {len(df)}")
print(f"Registros depois da limpeza: {len(df_limpo)}")
print(f"Duplicados restantes: {df_limpo['id'].duplicated().sum()}")
print(f"Datas ausentes preservadas: {df_limpo['data'].isna().sum()}")

display(df_limpo.head())

Registros antes da limpeza: 322
Registros depois da limpeza: 317
Duplicados restantes: 0
Datas ausentes preservadas: 6


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,,23640.97
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,,9447.52
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,,2891.48
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,,5636.46
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,,6641.24


## Regras determinísticas em escala

As mesmas regras do Nível 1 são reaplicadas sobre a base maior após a limpeza dos dados.

- Regra 1: fracionamento de operações;
- Regra 2: valor atípico em relação à mediana do cliente.

In [6]:
df_limpo["flag_fracionamento"] = False

df_com_data = df_limpo.dropna(subset=["data"]).copy()

resumo_dia = (
    df_com_data
    .groupby(["cliente_id", "data"])
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_dia_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max"),
    )
    .reset_index()
)

casos_fracionamento = resumo_dia[
    (resumo_dia["quantidade_operacoes"] >= 3)
    & (resumo_dia["soma_dia_brl"] > 50000)
    & (resumo_dia["maior_operacao_brl"] < 20000)
].copy()

for _, caso in casos_fracionamento.iterrows():
    mascara = (
        (df_limpo["cliente_id"] == caso["cliente_id"])
        & (df_limpo["data"] == caso["data"])
    )
    df_limpo.loc[mascara, "flag_fracionamento"] = True

print("Casos de fracionamento encontrados:")
display(casos_fracionamento)

Casos de fracionamento encontrados:


,cliente_id,data,quantidade_operacoes,soma_dia_brl,maior_operacao_brl
15,CLI-002,2026-05-01,4,64723.09,17998.60
28,CLI-003,2026-05-02,4,50846.72,18631.47
148,CLI-017,2026-03-08,4,64673.88,18761.22
276,CLI-029,2026-05-26,4,71297.68,19418.96


In [7]:
# ============================================================
# REGRA 2 — VALOR ATÍPICO
# ============================================================
#
# Regra:
# - cliente deve possuir pelo menos 4 operações;
# - valor da operação deve ser superior a
#   5 vezes a mediana do próprio cliente.
#
# A célula foi preparada para poder ser executada
# novamente sem gerar colunas _x e _y.
# ============================================================


# Remove colunas calculadas em execuções anteriores
colunas_derivadas = [
    "quantidade_operacoes",
    "mediana_valor_brl",
    "flag_valor_atipico",
    "quantidade_operacoes_x",
    "quantidade_operacoes_y",
    "mediana_valor_brl_x",
    "mediana_valor_brl_y",
]


df_limpo = df_limpo.drop(
    columns=[
        coluna
        for coluna in colunas_derivadas
        if coluna in df_limpo.columns
    ],
    errors="ignore"
)


# ============================================================
# ESTATÍSTICAS POR CLIENTE
# ============================================================

estatisticas_cliente = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        quantidade_operacoes=(
            "id",
            "count"
        ),
        mediana_valor_brl=(
            "valor_brl",
            "median"
        ),
    )
    .reset_index()
)


# ============================================================
# ADICIONA ESTATÍSTICAS AO DATAFRAME
# ============================================================

df_limpo = df_limpo.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left",
    validate="many_to_one"
)


# ============================================================
# APLICA A REGRA
# ============================================================

df_limpo["flag_valor_atipico"] = (
    (df_limpo["quantidade_operacoes"] >= 4)
    &
    (
        df_limpo["valor_brl"]
        > 5 * df_limpo["mediana_valor_brl"]
    )
)


# ============================================================
# OPERAÇÕES SINALIZADAS
# ============================================================

operacoes_valor_atipico = (
    df_limpo[
        df_limpo["flag_valor_atipico"]
    ][
        [
            "id",
            "cliente_id",
            "valor_brl",
            "mediana_valor_brl",
            "quantidade_operacoes",
            "flag_valor_atipico",
        ]
    ]
    .copy()
)


print(
    "Quantidade de operações com valor atípico:",
    len(operacoes_valor_atipico)
)

display(
    operacoes_valor_atipico
)

Quantidade de operações com valor atípico: 21


,id,cliente_id,valor_brl,mediana_valor_brl,quantidade_operacoes,flag_valor_atipico
0,OP-00133,CLI-014,23640.970,2308.410,11,True
18,OP-00197,CLI-021,15785.390,2832.545,10,True
19,OP-00129,CLI-014,13660.650,2308.410,11,True
24,OP-00253,CLI-026,21261.010,2032.930,12,True
52,OP-00219,CLI-023,41768.170,3241.160,12,True
82,OP-00008,CLI-001,25110.150,1609.155,10,True
110,OP-00049,CLI-005,11988.170,2144.180,11,True
117,OP-00310,CLI-013,28487.760,3146.225,10,True
121,OP-00312,CLI-022,30894.858,1909.890,11,True
126,OP-00316,CLI-024,68247.144,2740.780,11,True


In [8]:
resumo_clientes = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        sinalizacoes_fracionamento=("flag_fracionamento", "sum"),
        sinalizacoes_valor_atipico=("flag_valor_atipico", "sum"),
        volume_total_brl=("valor_brl", "sum"),
    )
    .reset_index()
)

resumo_clientes["total_sinalizacoes"] = (
    resumo_clientes["sinalizacoes_fracionamento"]
    + resumo_clientes["sinalizacoes_valor_atipico"]
)

top_10_clientes = (
    resumo_clientes
    .sort_values(
        ["total_sinalizacoes", "volume_total_brl"],
        ascending=[False, False],
    )
    .head(10)
    .reset_index(drop=True)
)

display(top_10_clientes)

,cliente_id,sinalizacoes_fracionamento,sinalizacoes_valor_atipico,volume_total_brl,total_sinalizacoes
0,CLI-029,4,0,191385.766,4
1,CLI-017,4,0,121391.370,4
2,CLI-002,4,0,107965.380,4
3,CLI-003,4,0,102241.090,4
4,CLI-014,0,3,80629.990,3
5,CLI-023,0,2,148535.016,2
6,CLI-028,0,2,88750.800,2
7,CLI-013,0,2,81730.990,2
8,CLI-005,0,2,64742.660,2
9,CLI-026,0,2,54729.280,2


## Ferramentas do agente

Antes da construção do agente, as ferramentas de investigação são testadas individualmente para verificar seu funcionamento.

In [9]:
# ============================================================
# FERRAMENTAS E AGENTE — VERSÃO ATUAL
# ============================================================

import sys
import importlib
from pathlib import Path


# ============================================================
# GARANTE ACESSO À PASTA nivel_2
# ============================================================

PASTA_ATUAL = Path.cwd()

if PASTA_ATUAL.name == "nivel_2":
    PASTA_NIVEL_2 = PASTA_ATUAL
else:
    PASTA_NIVEL_2 = PASTA_ATUAL / "nivel_2"


if str(PASTA_NIVEL_2) not in sys.path:
    sys.path.insert(
        0,
        str(PASTA_NIVEL_2)
    )


# ============================================================
# IMPORTA OS MÓDULOS
# ============================================================

import tools
import agente


# Recarrega para garantir que o Jupyter use
# a versão que acabamos de salvar no disco.
importlib.reload(tools)
importlib.reload(agente)


from tools import (
    historico_cliente,
    operacoes_do_dia,
    perfil_canal,
    operacoes_sinalizadas,
)

from agente import (
    investigar_cliente,
)


print(
    "Ferramentas e agente carregados com sucesso."
)

Ferramentas e agente carregados com sucesso.


In [10]:
# ============================================================
# TESTE DAS FERRAMENTAS
# ============================================================

cliente_teste = "CLI-029"


print("=" * 70)
print("1. HISTÓRICO DO CLIENTE")
print("=" * 70)

resultado_historico = historico_cliente(
    df_limpo,
    cliente_teste
)

print(
    resultado_historico
)


print("\n" + "=" * 70)
print("2. OPERAÇÕES SINALIZADAS")
print("=" * 70)

resultado_sinalizadas = operacoes_sinalizadas(
    df_limpo,
    cliente_teste
)

display(
    pd.DataFrame(
        resultado_sinalizadas
    )
)


print("\n" + "=" * 70)
print("3. OPERAÇÕES DO DIA")
print("=" * 70)

resultado_dia = operacoes_do_dia(
    df_limpo,
    cliente_teste,
    "2026-05-26"
)

display(
    pd.DataFrame(
        resultado_dia
    )
)


print("\n" + "=" * 70)
print("4. PERFIL DE CANAIS")
print("=" * 70)

resultado_canais = perfil_canal(
    df_limpo,
    cliente_teste
)

display(
    pd.DataFrame(
        resultado_canais
    )
)

1. HISTÓRICO DO CLIENTE
{'cliente_id': 'CLI-029', 'quantidade_operacoes': 16, 'volume_total_brl': 191385.766, 'mediana_valor_brl': 10337.485, 'maior_operacao_brl': 48045.636000000006, 'operacoes_fracionamento': 4, 'operacoes_valor_atipico': 0}

2. OPERAÇÕES SINALIZADAS


,id,data,valor_brl,canal,tipo,contraparte,flag_fracionamento,flag_valor_atipico
0,OP-00301,2026-05-26,19418.96,ted,saque,Orion Trading SA,True,False
1,OP-00300,2026-05-26,19138.59,ted,deposito,Solar Importacao LTDA,True,False
2,OP-00299,2026-05-26,14326.29,ted,transferencia_recebida,Rubi Importacao ME,True,False
3,OP-00302,2026-05-26,18413.84,pix,deposito,Cristal Atacado ME,True,False



3. OPERAÇÕES DO DIA


,id,data,valor_brl,canal,tipo,contraparte,flag_fracionamento,flag_valor_atipico
0,OP-00301,2026-05-26,19418.96,ted,saque,Orion Trading SA,True,False
1,OP-00300,2026-05-26,19138.59,ted,deposito,Solar Importacao LTDA,True,False
2,OP-00299,2026-05-26,14326.29,ted,transferencia_recebida,Rubi Importacao ME,True,False
3,OP-00302,2026-05-26,18413.84,pix,deposito,Cristal Atacado ME,True,False



4. PERFIL DE CANAIS


,canal,quantidade_operacoes,volume_total_brl,percentual_operacoes
0,boleto,3,15682.270,18.75
1,cartao,1,1666.850,6.25
2,especie,3,14978.620,18.75
3,pix,2,20877.860,12.50
4,ted,7,138180.166,43.75


## Teste do agente de investigação

O agente recebe apenas o identificador do cliente e decide autonomamente quais ferramentas utilizar durante a investigação.

In [13]:
# ============================================================
# TESTE DO AGENTE — UM ÚNICO CLIENTE
# ============================================================
#
# Antes da execução em lote, testamos somente um cliente
# para verificar:
#
# - seleção dinâmica de ferramentas;
# - resposta estruturada;
# - tokens;
# - latência;
# - custo estimado.
# ============================================================

cliente_teste_agente = "CLI-029"


print("=" * 70)
print(f"INICIANDO TESTE DO AGENTE — {cliente_teste_agente}")
print("=" * 70)


resultado_agente = investigar_cliente(
    df_limpo,
    cliente_teste_agente
)


print("\n" + "=" * 70)
print("RESULTADO DO AGENTE")
print("=" * 70)


print(
    "\nCliente:",
    resultado_agente["cliente_id"]
)


print(
    "\nResposta válida:",
    resultado_agente["resposta_valida"]
)


print(
    "\nFerramentas escolhidas:"
)

print(
    resultado_agente["ferramentas_usadas"]
)


print(
    "\nTotal de chamadas ao LLM:",
    len(
        resultado_agente["chamadas_llm"]
    )
)


print(
    "\nTokens totais:",
    resultado_agente["total_tokens"]
)


print(
    "\nLatência total:",
    round(
        resultado_agente[
            "latencia_total_segundos"
        ],
        3
    ),
    "segundos"
)


print(
    "\nCusto estimado:",
    f"US$ "
    f"{resultado_agente['custo_total_estimado_usd']:.8f}"
)


print("\nParecer estruturado:")

print(
    json.dumps(
        resultado_agente[
            "parecer_estruturado"
        ],
        ensure_ascii=False,
        indent=2
    )
)


print("\nDetalhes de cada chamada ao LLM:")

display(
    pd.DataFrame(
        resultado_agente[
            "chamadas_llm"
        ]
    )
)

INICIANDO TESTE DO AGENTE — CLI-029

RESULTADO DO AGENTE

Cliente: CLI-029

Resposta válida: True

Ferramentas escolhidas:
['operacoes_sinalizadas', 'operacoes_do_dia', 'historico_cliente']

Total de chamadas ao LLM: 4

Tokens totais: 6524

Latência total: 2.954 segundos

Custo estimado: US$ 0.00064050

Parecer estruturado:
{
  "cliente_id": "CLI-029",
  "nivel_risco": "médio",
  "tipologia_suspeita": "concentração de operações fracionadas em um único dia",
  "principais_evidencias": [
    "Quatro operações sinalizadas por fracionamento ocorrendo todas em 26/05/2026",
    "Operações envolvem valores entre R$14.326,29 e R$19.418,96, acima da mediana histórica de R$10.337,49",
    "Histórico geral do cliente mostra 16 operações, 4 fracionadas, sem ocorrências de valor atípico"
  ],
  "justificativa": "A concentração de quatro operações fracionadas em um único dia, representando 25% do total de operações do cliente, pode indicar tentativa de evitar limites de monitoramento. No entanto, nã

,numero_chamada,modelo,latencia_segundos,tokens_entrada,tokens_saida,tokens_total,custo_estimado_usd
0,1,openai/gpt-oss-20b,0.703723,909,64,973,0.000087
1,2,openai/gpt-oss-20b,0.619871,1310,92,1402,0.000126
2,3,openai/gpt-oss-20b,0.635252,1743,53,1796,0.000147
3,4,openai/gpt-oss-20b,0.994663,1890,463,2353,0.000281


In [15]:
# ============================================================
# EXECUÇÃO EM LOTE — TOP 10 CLIENTES
# ============================================================
#
# Esta etapa:
#
# 1. utiliza os 10 clientes priorizados pelas regras;
# 2. executa o agente para cada cliente;
# 3. salva um parecer estruturado por cliente;
# 4. registra métricas de CADA chamada ao LLM;
# 5. salva resultados incrementalmente para evitar perda
#    caso a API interrompa o processamento.
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd


# ============================================================
# CAMINHOS
# ============================================================

PASTA_ATUAL = Path.cwd()

if PASTA_ATUAL.name == "nivel_2":
    RAIZ_PROJETO = PASTA_ATUAL.parent
else:
    RAIZ_PROJETO = PASTA_ATUAL


PASTA_OUTPUTS = (
    RAIZ_PROJETO
    / "outputs"
)

PASTA_OUTPUTS.mkdir(
    parents=True,
    exist_ok=True
)


CAMINHO_LOTE_CSV = (
    PASTA_OUTPUTS
    / "lote_clientes.csv"
)

CAMINHO_LOTE_JSON = (
    PASTA_OUTPUTS
    / "lote_clientes.json"
)

CAMINHO_METRICAS = (
    PASTA_OUTPUTS
    / "metricas_execucao.csv"
)


# ============================================================
# CLIENTES DO LOTE
# ============================================================

clientes_lote = (
    top_10_clientes[
        "cliente_id"
    ]
    .tolist()
)


print(
    "Clientes selecionados para o lote:"
)

print(
    clientes_lote
)


# ============================================================
# INICIA UMA NOVA EXECUÇÃO
# ============================================================
#
# Estamos reconstruindo os outputs com o agente novo.
# Por isso, não utilizamos os resultados antigos.
# ============================================================

resultados_lote = []

metricas_chamadas = []


# ============================================================
# PROCESSAMENTO
# ============================================================

for indice, cliente_id in enumerate(
    clientes_lote,
    start=1
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"CLIENTE {indice}/"
        f"{len(clientes_lote)} "
        f"— {cliente_id}"
    )

    print(
        "=" * 70
    )


    try:

        inicio_cliente = (
            time.perf_counter()
        )


        resultado = (
            investigar_cliente(
                df_limpo,
                cliente_id
            )
        )


        tempo_cliente = (
            time.perf_counter()
            - inicio_cliente
        )


        parecer = resultado[
            "parecer_estruturado"
        ]


        # ====================================================
        # RESULTADO POR CLIENTE
        # ====================================================

        registro_cliente = {
            "cliente_id": (
                cliente_id
            ),

            "nivel_risco": (
                parecer[
                    "nivel_risco"
                ]
            ),

            "tipologia_suspeita": (
                parecer[
                    "tipologia_suspeita"
                ]
            ),

            "principais_evidencias": (
                json.dumps(
                    parecer[
                        "principais_evidencias"
                    ],
                    ensure_ascii=False
                )
            ),

            "justificativa": (
                parecer[
                    "justificativa"
                ]
            ),

            "recomendacao": (
                parecer[
                    "recomendacao"
                ]
            ),

            "resposta_valida": (
                resultado[
                    "resposta_valida"
                ]
            ),

            "ferramentas_usadas": (
                json.dumps(
                    resultado[
                        "ferramentas_usadas"
                    ],
                    ensure_ascii=False
                )
            ),

            "quantidade_chamadas_llm": (
                len(
                    resultado[
                        "chamadas_llm"
                    ]
                )
            ),

            "total_tokens": (
                resultado[
                    "total_tokens"
                ]
            ),

            "latencia_total_segundos": (
                resultado[
                    "latencia_total_segundos"
                ]
            ),

            "custo_total_estimado_usd": (
                resultado[
                    "custo_total_estimado_usd"
                ]
            ),

            "tempo_total_cliente_segundos": (
                tempo_cliente
            ),
        }


        resultados_lote.append(
            registro_cliente
        )


        # ====================================================
        # MÉTRICAS DE CADA CHAMADA
        # ====================================================

        for chamada in resultado[
            "chamadas_llm"
        ]:

            metricas_chamadas.append(
                {
                    "cliente_id": (
                        cliente_id
                    ),

                    "numero_chamada": (
                        chamada[
                            "numero_chamada"
                        ]
                    ),

                    "modelo": (
                        chamada[
                            "modelo"
                        ]
                    ),

                    "latencia_segundos": (
                        chamada[
                            "latencia_segundos"
                        ]
                    ),

                    "tokens_entrada": (
                        chamada[
                            "tokens_entrada"
                        ]
                    ),

                    "tokens_saida": (
                        chamada[
                            "tokens_saida"
                        ]
                    ),

                    "tokens_total": (
                        chamada[
                            "tokens_total"
                        ]
                    ),

                    "custo_estimado_usd": (
                        chamada[
                            "custo_estimado_usd"
                        ]
                    ),
                }
            )


        # ====================================================
        # SALVAMENTO INCREMENTAL
        # ====================================================
        #
        # Caso o processamento seja interrompido,
        # os clientes concluídos até aqui permanecem salvos.
        # ====================================================

        pd.DataFrame(
            resultados_lote
        ).to_csv(
            CAMINHO_LOTE_CSV,
            index=False,
            encoding="utf-8-sig"
        )


        pd.DataFrame(
            metricas_chamadas
        ).to_csv(
            CAMINHO_METRICAS,
            index=False,
            encoding="utf-8-sig"
        )


        with open(
            CAMINHO_LOTE_JSON,
            "w",
            encoding="utf-8"
        ) as arquivo:

            json.dump(
                resultados_lote,
                arquivo,
                ensure_ascii=False,
                indent=2
            )


        print(
            "Concluído:",
            cliente_id
        )

        print(
            "Risco:",
            parecer[
                "nivel_risco"
            ]
        )

        print(
            "Ferramentas:",
            resultado[
                "ferramentas_usadas"
            ]
        )

        print(
            "Chamadas ao LLM:",
            len(
                resultado[
                    "chamadas_llm"
                ]
            )
        )

        print(
            "Tokens:",
            resultado[
                "total_tokens"
            ]
        )

        print(
            "Latência:",
            f"{resultado['latencia_total_segundos']:.2f}s"
        )

        print(
            "Custo estimado:",
            f"US$ "
            f"{resultado['custo_total_estimado_usd']:.8f}"
        )


    except Exception as erro:

        print(
            f"ERRO ao investigar "
            f"{cliente_id}:"
        )

        print(
            erro
        )

        print(
            "O processamento continuará "
            "com o próximo cliente."
        )


    # Pequeno intervalo entre clientes
    # para reduzir pressão sobre a API.
    if indice < len(
        clientes_lote
    ):
        time.sleep(
            5
        )


# ============================================================
# DATAFRAMES FINAIS
# ============================================================

df_resultados_lote = (
    pd.DataFrame(
        resultados_lote
    )
)

df_metricas_chamadas = (
    pd.DataFrame(
        metricas_chamadas
    )
)


# ============================================================
# RESULTADO
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "LOTE FINALIZADO"
)

print(
    "=" * 70
)


print(
    "Clientes concluídos:",
    len(
        df_resultados_lote
    ),
    "/",
    len(
        clientes_lote
    )
)


print(
    "\nArquivos:"
)

print(
    "lote_clientes.csv:",
    CAMINHO_LOTE_CSV.exists()
)

print(
    "lote_clientes.json:",
    CAMINHO_LOTE_JSON.exists()
)

print(
    "metricas_execucao.csv:",
    CAMINHO_METRICAS.exists()
)


display(
    df_resultados_lote
)

Clientes selecionados para o lote:
['CLI-029', 'CLI-017', 'CLI-002', 'CLI-003', 'CLI-014', 'CLI-023', 'CLI-028', 'CLI-013', 'CLI-005', 'CLI-026']

CLIENTE 1/10 — CLI-029
Concluído: CLI-029
Risco: médio
Ferramentas: ['operacoes_sinalizadas', 'operacoes_do_dia', 'historico_cliente']
Chamadas ao LLM: 4
Tokens: 6776
Latência: 2.52s
Custo estimado: US$ 0.00071610

CLIENTE 2/10 — CLI-017
Concluído: CLI-017
Risco: médio
Ferramentas: ['operacoes_sinalizadas', 'operacoes_do_dia', 'historico_cliente']
Chamadas ao LLM: 4
Tokens: 6647
Latência: 32.52s
Custo estimado: US$ 0.00066210

CLIENTE 3/10 — CLI-002
Concluído: CLI-002
Risco: médio
Ferramentas: ['operacoes_sinalizadas', 'operacoes_do_dia', 'historico_cliente']
Chamadas ao LLM: 4
Tokens: 6952
Latência: 48.28s
Custo estimado: US$ 0.00072705

CLIENTE 4/10 — CLI-003
Concluído: CLI-003
Risco: médio
Ferramentas: ['operacoes_sinalizadas', 'operacoes_do_dia', 'historico_cliente']
Chamadas ao LLM: 4
Tokens: 6757
Latência: 37.32s
Custo estimado: US$ 0.

,cliente_id,nivel_risco,tipologia_suspeita,principais_evidencias,justificativa,recomendacao,resposta_valida,ferramentas_usadas,quantidade_chamadas_llm,total_tokens,latencia_total_segundos,custo_total_estimado_usd,tempo_total_cliente_segundos
0,CLI-029,médio,concentração de operações fracionadas em um ún...,"[""4 operações sinalizadas por fracionamento em...","O cliente possui 16 operações no histórico, co...","Monitorar as transações futuras do cliente, so...",True,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4,6776,2.516631,0.000716,2.525602
1,CLI-017,médio,concentração de operações em um único dia com ...,"[""Quatro operações sinalizadas (OP-00303, OP-0...",A concentração de quatro transações de fracion...,"Monitorar as transações futuras do cliente, so...",True,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4,6647,32.524283,0.000662,32.532663
2,CLI-002,médio,concentração de operações fracionadas em curto...,"[""Quatro operações fracionadas foram registrad...",O cliente apresenta um padrão de fracionamento...,Recomenda-se monitoramento contínuo das operaç...,True,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4,6952,48.283530,0.000727,48.291910
3,CLI-003,médio,fracionamento de valor em dia único,"[""Quatro operações sinalizadas por fracionamen...",O histórico do cliente mostra 14 operações no ...,"Monitorar as operações futuras do cliente, sol...",True,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4,6757,37.315225,0.000693,37.323192
4,CLI-014,baixo,valor_atipico em operações de alto valor,"[""Três operações sinalizadas por valor atípico...",As operações sinalizadas apresentam valores su...,Manter monitoramento contínuo das transações d...,True,"[""operacoes_sinalizadas"", ""historico_cliente""]",3,4114,25.021939,0.000439,25.026640
5,CLI-023,baixo,valor_atipico em depósito e transferência,"[""Operação de depósito em espécie de R$ 41.768...",O histórico financeiro do cliente mostra 12 op...,Manter monitoramento contínuo das transações d...,True,"[""operacoes_sinalizadas"", ""historico_cliente""]",3,4262,17.298541,0.000508,17.303323
6,CLI-028,médio,Transferências de alto valor concentradas em u...,"[""Operação OP-00308: transferência enviada de ...","O cliente possui 12 operações no histórico, co...",Recomenda-se monitoramento contínuo das transa...,True,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4,6432,30.080218,0.000713,30.088135
7,CLI-013,médio,concentração de operações de alto valor em um ...,"[""Operação OP-00310: saque de R$28.487,76 em 2...",As duas operações sinalizadas representam a ma...,Recomenda-se monitoramento contínuo das operaç...,True,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4,6020,30.812805,0.000647,30.820819
8,CLI-005,baixo,valor_atipico em operações de pagamento e tran...,"[""Operação OP-00049: transferência recebida de...",As duas operações sinalizadas são as maiores d...,"Monitorar as próximas operações do cliente, so...",True,"[""operacoes_sinalizadas"", ""historico_cliente""]",3,4118,22.137759,0.000456,22.142648
9,CLI-026,médio,valor atípico em transações de alto valor,"[""Operação OP-00253: pagamento via PIX de R$21...",As duas operações sinalizadas são de alto valo...,Manter monitoramento contínuo das transações d...,True,"[""operacoes_sinalizadas"", ""historico_cliente""]",3,4117,15.109167,0.000467,25.599235


In [17]:
# ============================================================
# ANÁLISE DOS RESULTADOS E MÉTRICAS DO LOTE
# ============================================================
#
# Os resultados do lote já estão estruturados.
# Não é necessário fazer json.loads() do parecer novamente.
#
# Esta etapa analisa com pandas:
#
# - clientes processados;
# - respostas válidas;
# - quantidade de chamadas ao LLM;
# - tokens;
# - latência;
# - custo estimado;
# - ferramentas escolhidas.
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# RECUPERA OS DATAFRAMES, SE NECESSÁRIO
# ============================================================

PASTA_ATUAL = Path.cwd()

if PASTA_ATUAL.name == "nivel_2":
    RAIZ_PROJETO = PASTA_ATUAL.parent
else:
    RAIZ_PROJETO = PASTA_ATUAL

PASTA_OUTPUTS = RAIZ_PROJETO / "outputs"


# Se o DataFrame não estiver mais na memória,
# carrega o arquivo salvo pelo lote.
if "df_resultados_lote" not in globals():

    df_resultados_lote = pd.read_csv(
        PASTA_OUTPUTS / "lote_clientes.csv"
    )


if "df_metricas_chamadas" not in globals():

    df_metricas_chamadas = pd.read_csv(
        PASTA_OUTPUTS / "metricas_execucao.csv"
    )


# ============================================================
# VERIFICAÇÃO DOS RESULTADOS
# ============================================================

quantidade_clientes = len(
    df_resultados_lote
)


# Trata resposta_valida tanto como booleano
# quanto como texto vindo de CSV.
respostas_validas = (
    df_resultados_lote[
        "resposta_valida"
    ]
    .astype(str)
    .str.lower()
    .eq("true")
)


quantidade_respostas_validas = int(
    respostas_validas.sum()
)


quantidade_chamadas = len(
    df_metricas_chamadas
)


print("=" * 70)
print("VERIFICAÇÃO DO LOTE")
print("=" * 70)

print(
    f"Clientes processados: "
    f"{quantidade_clientes}/10"
)

print(
    f"Respostas válidas: "
    f"{quantidade_respostas_validas}/"
    f"{quantidade_clientes}"
)

print(
    f"Chamadas ao LLM: "
    f"{quantidade_chamadas}"
)


# ============================================================
# TOKENS
# ============================================================

tokens_entrada_total = int(
    df_metricas_chamadas[
        "tokens_entrada"
    ].sum()
)

tokens_saida_total = int(
    df_metricas_chamadas[
        "tokens_saida"
    ].sum()
)

tokens_total = int(
    df_metricas_chamadas[
        "tokens_total"
    ].sum()
)


media_tokens_por_chamada = (
    df_metricas_chamadas[
        "tokens_total"
    ].mean()
)


media_tokens_por_cliente = (
    df_resultados_lote[
        "total_tokens"
    ].mean()
)


# ============================================================
# LATÊNCIA
# ============================================================

latencia_total = (
    df_metricas_chamadas[
        "latencia_segundos"
    ].sum()
)


latencia_media_chamada = (
    df_metricas_chamadas[
        "latencia_segundos"
    ].mean()
)


latencia_media_cliente = (
    df_resultados_lote[
        "latencia_total_segundos"
    ].mean()
)


maior_latencia_chamada = (
    df_metricas_chamadas[
        "latencia_segundos"
    ].max()
)


# ============================================================
# CUSTO
# ============================================================

custo_total = (
    df_metricas_chamadas[
        "custo_estimado_usd"
    ].sum()
)


custo_medio_chamada = (
    df_metricas_chamadas[
        "custo_estimado_usd"
    ].mean()
)


custo_medio_cliente = (
    df_resultados_lote[
        "custo_total_estimado_usd"
    ].mean()
)


# ============================================================
# RESUMO GERAL
# ============================================================

resumo_metricas = pd.DataFrame(
    [
        {
            "metrica": "Clientes processados",
            "valor": quantidade_clientes
        },
        {
            "metrica": "Respostas válidas",
            "valor": quantidade_respostas_validas
        },
        {
            "metrica": "Chamadas ao LLM",
            "valor": quantidade_chamadas
        },
        {
            "metrica": "Tokens de entrada",
            "valor": tokens_entrada_total
        },
        {
            "metrica": "Tokens de saída",
            "valor": tokens_saida_total
        },
        {
            "metrica": "Tokens totais",
            "valor": tokens_total
        },
        {
            "metrica": "Média de tokens por chamada",
            "valor": round(
                media_tokens_por_chamada,
                2
            )
        },
        {
            "metrica": "Média de tokens por cliente",
            "valor": round(
                media_tokens_por_cliente,
                2
            )
        },
        {
            "metrica": "Latência total das chamadas (s)",
            "valor": round(
                latencia_total,
                3
            )
        },
        {
            "metrica": "Latência média por chamada (s)",
            "valor": round(
                latencia_media_chamada,
                3
            )
        },
        {
            "metrica": "Latência média por cliente (s)",
            "valor": round(
                latencia_media_cliente,
                3
            )
        },
        {
            "metrica": "Maior latência de uma chamada (s)",
            "valor": round(
                maior_latencia_chamada,
                3
            )
        },
        {
            "metrica": "Custo estimado total (USD)",
            "valor": round(
                custo_total,
                8
            )
        },
        {
            "metrica": "Custo médio por chamada (USD)",
            "valor": round(
                custo_medio_chamada,
                8
            )
        },
        {
            "metrica": "Custo médio por cliente (USD)",
            "valor": round(
                custo_medio_cliente,
                8
            )
        },
    ]
)


print("\n" + "=" * 70)
print("RESUMO DAS MÉTRICAS")
print("=" * 70)

display(
    resumo_metricas
)


# ============================================================
# MÉTRICAS POR CLIENTE
# ============================================================

metricas_por_cliente = (
    df_metricas_chamadas
    .groupby(
        "cliente_id",
        as_index=False
    )
    .agg(
        chamadas_llm=(
            "numero_chamada",
            "count"
        ),
        tokens_entrada=(
            "tokens_entrada",
            "sum"
        ),
        tokens_saida=(
            "tokens_saida",
            "sum"
        ),
        tokens_total=(
            "tokens_total",
            "sum"
        ),
        latencia_total_segundos=(
            "latencia_segundos",
            "sum"
        ),
        custo_estimado_usd=(
            "custo_estimado_usd",
            "sum"
        ),
    )
)


metricas_por_cliente = (
    metricas_por_cliente
    .merge(
        df_resultados_lote[
            [
                "cliente_id",
                "nivel_risco",
                "ferramentas_usadas",
                "resposta_valida",
            ]
        ],
        on="cliente_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "tokens_total",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("MÉTRICAS POR CLIENTE")
print("=" * 70)

display(
    metricas_por_cliente
)


# ============================================================
# SELEÇÃO DINÂMICA DE FERRAMENTAS
# ============================================================

print("\n" + "=" * 70)
print("FERRAMENTAS ESCOLHIDAS POR CLIENTE")
print("=" * 70)

display(
    df_resultados_lote[
        [
            "cliente_id",
            "nivel_risco",
            "ferramentas_usadas",
            "quantidade_chamadas_llm",
        ]
    ]
)

VERIFICAÇÃO DO LOTE
Clientes processados: 10/10
Respostas válidas: 10/10
Chamadas ao LLM: 36

RESUMO DAS MÉTRICAS


,metrica,valor
0,Clientes processados,10.000000
1,Respostas válidas,10.000000
2,Chamadas ao LLM,36.000000
3,Tokens de entrada,48128.000000
4,Tokens de saída,8067.000000
5,Tokens totais,56195.000000
6,Média de tokens por chamada,1560.970000
7,Média de tokens por cliente,5619.500000
8,Latência total das chamadas (s),261.100000
9,Latência média por chamada (s),7.253000



MÉTRICAS POR CLIENTE


,cliente_id,chamadas_llm,tokens_entrada,tokens_saida,tokens_total,latencia_total_segundos,custo_estimado_usd,nivel_risco,ferramentas_usadas,resposta_valida
0,CLI-002,4,6038,914,6952,48.283530,0.000727,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",True
1,CLI-029,4,5852,924,6776,2.516631,0.000716,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",True
2,CLI-003,4,5928,829,6757,37.315225,0.000693,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",True
3,CLI-017,4,5920,727,6647,32.524283,0.000662,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",True
4,CLI-028,4,5406,1026,6432,30.080218,0.000713,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",True
5,CLI-013,4,5152,868,6020,30.812805,0.000647,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",True
6,CLI-023,3,3423,839,4262,17.298541,0.000508,baixo,"[""operacoes_sinalizadas"", ""historico_cliente""]",True
7,CLI-005,3,3462,656,4118,22.137759,0.000456,baixo,"[""operacoes_sinalizadas"", ""historico_cliente""]",True
8,CLI-026,3,3413,704,4117,15.109167,0.000467,médio,"[""operacoes_sinalizadas"", ""historico_cliente""]",True
9,CLI-014,3,3534,580,4114,25.021939,0.000439,baixo,"[""operacoes_sinalizadas"", ""historico_cliente""]",True



FERRAMENTAS ESCOLHIDAS POR CLIENTE


,cliente_id,nivel_risco,ferramentas_usadas,quantidade_chamadas_llm
0,CLI-029,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4
1,CLI-017,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4
2,CLI-002,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4
3,CLI-003,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4
4,CLI-014,baixo,"[""operacoes_sinalizadas"", ""historico_cliente""]",3
5,CLI-023,baixo,"[""operacoes_sinalizadas"", ""historico_cliente""]",3
6,CLI-028,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4
7,CLI-013,médio,"[""operacoes_sinalizadas"", ""operacoes_do_dia"", ...",4
8,CLI-005,baixo,"[""operacoes_sinalizadas"", ""historico_cliente""]",3
9,CLI-026,médio,"[""operacoes_sinalizadas"", ""historico_cliente""]",3


In [ ]:
# ============================================================
# VALIDAÇÃO FINAL E PERSISTÊNCIA DOS OUTPUTS DO LOTE
# ============================================================
#
# Esta etapa valida os resultados produzidos pelo agente,
# persiste os arquivos finais e confirma que os campos
# obrigatórios foram salvos corretamente.
#
# Também garante que campos estruturados, como evidências
# e ferramentas utilizadas, sejam representados
# adequadamente no JSON final.
#
# Esta célula NÃO realiza novas chamadas ao LLM.
# ============================================================

import json
from pathlib import Path

import pandas as pd


# ============================================================
# CAMINHOS
# ============================================================

PASTA_ATUAL = Path.cwd()

if PASTA_ATUAL.name == "nivel_2":
    RAIZ_PROJETO = PASTA_ATUAL.parent
else:
    RAIZ_PROJETO = PASTA_ATUAL


PASTA_OUTPUTS = (
    RAIZ_PROJETO
    / "outputs"
)

PASTA_OUTPUTS.mkdir(
    parents=True,
    exist_ok=True
)


CAMINHO_LOTE_CSV = (
    PASTA_OUTPUTS
    / "lote_clientes.csv"
)

CAMINHO_LOTE_JSON = (
    PASTA_OUTPUTS
    / "lote_clientes.json"
)

CAMINHO_METRICAS = (
    PASTA_OUTPUTS
    / "metricas_execucao.csv"
)


# ============================================================
# VERIFICA SE OS RESULTADOS AINDA ESTÃO NA MEMÓRIA
# ============================================================

if "df_resultados_lote" not in globals():
    raise RuntimeError(
        "df_resultados_lote não está na memória. "
        "Não reinicie o kernel e me avise antes de continuar."
    )


if "df_metricas_chamadas" not in globals():
    raise RuntimeError(
        "df_metricas_chamadas não está na memória. "
        "Não reinicie o kernel e me avise antes de continuar."
    )


# ============================================================
# CÓPIA DE SEGURANÇA
# ============================================================

df_lote_corrigido = (
    df_resultados_lote
    .copy()
)


# ============================================================
# COLUNAS OBRIGATÓRIAS
# ============================================================

colunas_obrigatorias = [
    "cliente_id",
    "nivel_risco",
    "tipologia_suspeita",
    "principais_evidencias",
    "justificativa",
    "recomendacao",
    "resposta_valida",
    "ferramentas_usadas",
    "quantidade_chamadas_llm",
    "total_tokens",
    "latencia_total_segundos",
]


colunas_ausentes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_lote_corrigido.columns
]


if colunas_ausentes:
    raise RuntimeError(
        "Colunas ausentes no resultado em memória: "
        + ", ".join(colunas_ausentes)
    )


# ============================================================
# VERIFICAÇÃO ANTES DE SALVAR
# ============================================================

print("=" * 70)
print("RESULTADOS EM MEMÓRIA")
print("=" * 70)


print(
    "Quantidade de clientes:",
    len(df_lote_corrigido)
)


display(
    df_lote_corrigido[
        [
            "cliente_id",
            "nivel_risco",
            "resposta_valida",
            "total_tokens",
            "latencia_total_segundos",
        ]
    ]
)


print(
    "\nValores ausentes nas colunas principais:"
)

print(
    df_lote_corrigido[
        [
            "cliente_id",
            "nivel_risco",
            "tipologia_suspeita",
            "justificativa",
            "recomendacao",
        ]
    ]
    .isna()
    .sum()
)


# ============================================================
# PROTEÇÃO
# ============================================================
#
# Não queremos salvar um arquivo quebrado.
# ============================================================

if len(df_lote_corrigido) != 10:
    raise RuntimeError(
        f"Esperados 10 clientes, mas existem "
        f"{len(df_lote_corrigido)}."
    )


if df_lote_corrigido[
    "nivel_risco"
].isna().any():

    raise RuntimeError(
        "Existem valores vazios em nivel_risco. "
        "O CSV não será sobrescrito."
    )


if df_lote_corrigido[
    "justificativa"
].isna().any():

    raise RuntimeError(
        "Existem justificativas vazias. "
        "O CSV não será sobrescrito."
    )


# ============================================================
# SALVA O CSV CORRETO
# ============================================================

df_lote_corrigido.to_csv(
    CAMINHO_LOTE_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# PREPARA O JSON ESTRUTURADO
# ============================================================
#
# No CSV, principais_evidencias é armazenado como texto JSON.
# No arquivo .json, voltamos a transformá-lo em lista.
# ============================================================

registros_json = json.loads(
    df_lote_corrigido.to_json(
        orient="records",
        force_ascii=False
    )
)


for registro in registros_json:

    evidencias = registro.get(
        "principais_evidencias"
    )

    if isinstance(
        evidencias,
        str
    ):

        try:

            registro[
                "principais_evidencias"
            ] = json.loads(
                evidencias
            )

        except json.JSONDecodeError:

            registro[
                "principais_evidencias"
            ] = [
                evidencias
            ]


    ferramentas = registro.get(
        "ferramentas_usadas"
    )

    if isinstance(
        ferramentas,
        str
    ):

        try:

            registro[
                "ferramentas_usadas"
            ] = json.loads(
                ferramentas
            )

        except json.JSONDecodeError:

            registro[
                "ferramentas_usadas"
            ] = [
                ferramentas
            ]


# ============================================================
# SALVA JSON CORRETO
# ============================================================

with open(
    CAMINHO_LOTE_JSON,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        registros_json,
        arquivo,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# GARANTE QUE AS MÉTRICAS CONTINUEM SALVAS
# ============================================================

df_metricas_chamadas.to_csv(
    CAMINHO_METRICAS,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# RELÊ O CSV DO DISCO
# ============================================================
#
# Essa é a verificação mais importante.
# Não estamos olhando a variável da memória;
# estamos lendo exatamente o arquivo que confronto.py usará.
# ============================================================

df_verificacao_disco = pd.read_csv(
    CAMINHO_LOTE_CSV
)


print(
    "\n"
    + "=" * 70
)

print(
    "VERIFICAÇÃO DO ARQUIVO SALVO NO DISCO"
)

print(
    "=" * 70
)


display(
    df_verificacao_disco[
        [
            "cliente_id",
            "nivel_risco",
            "resposta_valida",
            "justificativa",
        ]
    ]
)


print(
    "\nValores ausentes após reler o CSV:"
)

print(
    df_verificacao_disco[
        [
            "cliente_id",
            "nivel_risco",
            "justificativa",
        ]
    ]
    .isna()
    .sum()
)


print(
    "\nArquivos:"
)

print(
    "lote_clientes.csv:",
    CAMINHO_LOTE_CSV.exists()
)

print(
    "lote_clientes.json:",
    CAMINHO_LOTE_JSON.exists()
)

print(
    "metricas_execucao.csv:",
    CAMINHO_METRICAS.exists()
)

RESULTADOS EM MEMÓRIA
Quantidade de clientes: 10


,cliente_id,nivel_risco,resposta_valida,total_tokens,latencia_total_segundos
0,CLI-029,médio,True,6776,2.516631
1,CLI-017,médio,True,6647,32.524283
2,CLI-002,médio,True,6952,48.283530
3,CLI-003,médio,True,6757,37.315225
4,CLI-014,baixo,True,4114,25.021939
5,CLI-023,baixo,True,4262,17.298541
6,CLI-028,médio,True,6432,30.080218
7,CLI-013,médio,True,6020,30.812805
8,CLI-005,baixo,True,4118,22.137759
9,CLI-026,médio,True,4117,15.109167



Valores ausentes nas colunas principais:
cliente_id            0
nivel_risco           0
tipologia_suspeita    0
justificativa         0
recomendacao          0
dtype: int64

VERIFICAÇÃO DO ARQUIVO SALVO NO DISCO


,cliente_id,nivel_risco,resposta_valida,justificativa
0,CLI-029,médio,True,"O cliente possui 16 operações no histórico, co..."
1,CLI-017,médio,True,A concentração de quatro transações de fracion...
2,CLI-002,médio,True,O cliente apresenta um padrão de fracionamento...
3,CLI-003,médio,True,O histórico do cliente mostra 14 operações no ...
4,CLI-014,baixo,True,As operações sinalizadas apresentam valores su...
5,CLI-023,baixo,True,O histórico financeiro do cliente mostra 12 op...
6,CLI-028,médio,True,"O cliente possui 12 operações no histórico, co..."
7,CLI-013,médio,True,As duas operações sinalizadas representam a ma...
8,CLI-005,baixo,True,As duas operações sinalizadas são as maiores d...
9,CLI-026,médio,True,As duas operações sinalizadas são de alto valo...



Valores ausentes após reler o CSV:
cliente_id       0
nivel_risco      0
justificativa    0
dtype: int64

Arquivos:
lote_clientes.csv: True
lote_clientes.json: True
metricas_execucao.csv: True


## Confronto entre regras determinísticas e agente

A etapa final compara a classificação de risco produzida pelas regras
determinísticas com a classificação do agente para os 10 clientes
priorizados.

O critério determinístico adotado foi:

- **baixo:** nenhuma regra acionada;
- **médio:** um único evento determinístico;
- **alto:** dois ou mais eventos ou ocorrência das duas tipologias.

Esse critério representa apenas uma priorização de triagem. Ele não deve ser
interpretado como confirmação de lavagem de dinheiro ou de qualquer ilícito.

Além da taxa de concordância, as divergências são analisadas individualmente,
pois as regras são simples e podem gerar classificações excessivamente
sensíveis.

In [20]:
# ============================================================
# CONFRONTO — REGRAS DETERMINÍSTICAS x AGENTE
# ============================================================

import importlib
import confronto

# Recarrega o módulo para garantir que o notebook
# utilize a versão atual salva em confronto.py.
importlib.reload(confronto)

from confronto import executar_confronto


confronto_final, resumo_confronto = (
    executar_confronto()
)


print(
    "\nResumo do confronto:"
)

print(
    json.dumps(
        resumo_confronto,
        ensure_ascii=False,
        indent=2
    )
)

CONFRONTO — REGRAS DETERMINÍSTICAS x AGENTE

RESULTADO POR CLIENTE:

cliente_id  eventos_fracionamento  eventos_valor_atipico  total_eventos risco_deterministico risco_agente resultado_confronto
   CLI-029                      1                      0              1                médio        médio         concordante
   CLI-017                      1                      0              1                médio        médio         concordante
   CLI-002                      1                      0              1                médio        médio         concordante
   CLI-003                      1                      0              1                médio        médio         concordante
   CLI-014                      0                      3              3                 alto        baixo          divergente
   CLI-023                      0                      2              2                 alto        baixo          divergente
   CLI-028                      0                

In [21]:
# ============================================================
# RESUMO VISUAL DO CONFRONTO
# ============================================================

colunas_resumo = [
    "cliente_id",
    "eventos_fracionamento",
    "eventos_valor_atipico",
    "risco_deterministico",
    "risco_agente",
    "resultado_confronto",
    "quem_parece_mais_adequado",
]


display(
    confronto_final[
        colunas_resumo
    ]
)

,cliente_id,eventos_fracionamento,eventos_valor_atipico,risco_deterministico,risco_agente,resultado_confronto,quem_parece_mais_adequado
0,CLI-029,1,0,médio,médio,concordante,-
1,CLI-017,1,0,médio,médio,concordante,-
2,CLI-002,1,0,médio,médio,concordante,-
3,CLI-003,1,0,médio,médio,concordante,-
4,CLI-014,0,3,alto,baixo,divergente,nenhum dos dois isoladamente
5,CLI-023,0,2,alto,baixo,divergente,nenhum dos dois isoladamente
6,CLI-028,0,2,alto,médio,divergente,"agente, com revisão humana"
7,CLI-013,0,2,alto,médio,divergente,"agente na classificação, com ressalva na justi..."
8,CLI-005,0,2,alto,baixo,divergente,nenhum dos dois isoladamente
9,CLI-026,0,2,alto,médio,divergente,"agente, com revisão humana"


### Conclusão do Nível 2

O confronto resultou em **40% de concordância** entre a classificação
determinística e o agente: 4 dos 10 clientes receberam o mesmo nível de risco.

Os quatro casos de fracionamento foram classificados como risco médio tanto
pelas regras quanto pelo agente. As seis divergências ocorreram entre clientes
com múltiplas operações classificadas como valor atípico.

Nesses casos, a regra determinística tende a elevar automaticamente o risco
quando existem dois ou mais eventos, enquanto o agente utiliza ferramentas
adicionais para considerar o contexto do cliente. A análise mostrou, porém,
que a decisão do agente também não deve ser aceita automaticamente.

Nos clientes `CLI-014`, `CLI-023` e `CLI-005`, a classificação determinística
como alto parece excessivamente sensível, mas a classificação do agente como
baixo também parece permissiva. Uma priorização intermediária seria mais
proporcional às evidências disponíveis.

Nos clientes `CLI-028` e `CLI-026`, a classificação de risco médio produzida
pelo agente parece mais proporcional do que elevar automaticamente o risco
pela repetição da mesma regra.

O caso `CLI-013` também demonstra uma limitação importante do uso de LLMs:
embora a classificação de risco médio pareça razoável, a justificativa
introduziu uma hipótese sobre possível intenção de ocultação que não é
comprovada pelos dados. Isso reforça a necessidade de validação humana e de
prompts restritivos.

Assim, a taxa de concordância isoladamente não é suficiente para avaliar o
agente. O principal benefício do agente está em adicionar contexto às regras
determinísticas, enquanto as regras continuam importantes como mecanismo
objetivo e reproduzível de triagem.